<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Algorithmic Trading
## Chapter 5 · Static EOD Data and Data Engineering

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


### Notebook Goals

This notebook walks through the Chapter 5 data-engineering helpers and adds interactive views for the feature panel.

- Load the static end-of-day panel via `load_eod_panel`.
- Build a small feature panel for SPY and inspect its distributions.
- Visualise missing data and large moves to develop data intuition.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")  # set notebook-wide plotting style
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("svg")
except Exception:
    pass
plt.rcParams.update({"figure.dpi": 300})

PROJECT_ROOT = Path("..").resolve()  # project root relative to notebooks
CODE_DIR = PROJECT_ROOT / "code"  # folder with chapter scripts
if str(CODE_DIR) not in sys.path:  # ensure scripts are importable
    sys.path.insert(0, str(CODE_DIR))

import ch05_eod_engineering as ch05  # EOD data helpers


### 1. Load the End-of-Day Panel

We pull the price panel into memory and reproduce the summary numbers used throughout the book.

In [ ]:
panel = ch05.load_eod_panel()  # multi-asset price panel
panel_stats = ch05.describe_panel(panel)
panel.head()


In [ ]:
panel_stats


### 2. Feature Panel for SPY

We focus on SPY and build a small feature panel that captures returns, rolling means, volatility, and z-scores.

In [ ]:
log_ret_spy = ch05.load_spy_log_returns()
features_spy = ch05.build_feature_panel(log_ret_spy)
features_spy.tail()


### 3. Visualising Features and Large Moves

The next plot shows the z-score feature and highlights days with unusually large log-returns.

In [ ]:
n_large = ch05.count_large_moves(log_ret_spy, threshold=0.045)

fig, ax = plt.subplots(figsize=(10.0, 4.0))
ax.plot(features_spy.index, features_spy["z_score"], label="z-score")
ax.axhline(0.0, color="black", lw=0.8)
ax.set_ylabel("z-score")
ax.set_title(
    f"SPY z-score feature with |r_t| > 4.5% highlighted (count = {n_large})",
)

mask_large = np.abs(log_ret_spy) > 0.045
ax.scatter(
    log_ret_spy.index[mask_large],
    features_spy.loc[mask_large, "z_score"],
    color="tab:red",
    s=12,
    label="large moves",
)
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


### Takeaways

- A single well-structured EOD panel can power many experiments across the book.
- Feature panels built from log-returns (means, volatility, z-scores) provide natural inputs for strategies and models.
- Visualising extreme moves early on helps you spot data issues and think about risk management in realistic terms.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">